In [1]:
# import packages
import pandas as pd

## Load Data

In [6]:
data = pd.read_csv("dataset/final_df.csv")

data.head()

,Order Date,Purchase Price Per Unit,Quantity,Shipping Address State,Title,ASIN/ISBN (Product Code),Survey ResponseID,Category
0,2018-12-04,7.98,1,NJ,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,R_01vNIayewjIIKMF,Electronics
1,2018-12-22,13.99,1,NJ,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,R_01vNIayewjIIKMF,Electronics
2,2018-12-24,8.99,1,NJ,NaN,B078JZTFN3,R_01vNIayewjIIKMF,NaN
3,2018-12-25,10.45,1,NJ,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,R_01vNIayewjIIKMF,Kitchen & Dining
4,2018-12-25,10.00,1,NJ,Proraso Shaving Cream for Men,B00837ZOI0,R_01vNIayewjIIKMF,Beauty & Personal Care


## Clean Data

In [7]:
# drop unnecessary columns
df = data.drop(columns=["Order Date", "Shipping Address State", "Survey ResponseID"])

# remove null / NA values
df = df.dropna()

df


,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category
0,7.98,1,SanDisk Ultra 16GB Class 10 SDHC UHS-I Memory ...,B0143RTB1E,Electronics
1,13.99,1,Betron BS10 Earphones Wired Headphones in Ear ...,B01MA1MJ6H,Electronics
3,10.45,1,Perfecto Stainless Steel Shaving Bowl. Durable...,B06XWF9HML,Kitchen & Dining
4,10.00,1,Proraso Shaving Cream for Men,B00837ZOI0,Beauty & Personal Care
5,10.99,1,Micro USB Cable Android Charger - Syncwire [2-...,B01GFB2E9M,Computer & Accessories
...,...,...,...,...,...
1392015,6.99,1,Tanner's Tasty Paste Vanilla Bling - Anticavit...,B015ZRTHVA,"Health, Household & Personal Care"
1392016,15.99,1,Sinland Microfiber Cleaning Cloth Dish Cloth K...,B00QGCXPRG,"Health, Household & Personal Care"
1392017,6.99,4,Tanner's Tasty Paste Vanilla Bling - Anticavit...,B015ZRTHVA,"Health, Household & Personal Care"
1392018,6.99,4,Tanner's Tasty Paste Vanilla Bling - Anticavit...,B015ZRTHVA,"Health, Household & Personal Care"


## Data Preprocessing

In [8]:
# finding the most common item in order to compare for competition
from collections import Counter 
import nltk

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

# to remove stopwords and symbols and numbers
def is_valid_word(word):
    return word.isalpha() and len(word) >= 3

# split title into list of words
df_split_title = df["Title"].str.split()

# define noun tags
NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}

# process each title to extract nouns without modifying the original df
nouns_count = Counter(
    word 
    for title in df_split_title
    for word, pos in nltk.pos_tag(title)  # POS tag each word in each title
    if is_valid_word(word) and pos in NOUN_TAGS  # keep only nouns
)

nouns_count


[nltk_data] Downloading package punkt to /Users/raisie/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/raisie/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


Counter({'Pack': 86249,
         'Black': 65305,
         'Count': 57772,
         'Set': 53579,
         'Women': 49854,
         'Ounce': 49240,
         'Hair': 42933,
         'Gift': 42417,
         'Card': 40799,
         'Free': 39532,
         'White': 39414,
         'Dog': 39117,
         'Amazon': 38509,
         'Kids': 38448,
         'Cat': 37872,
         'Case': 37738,
         'Natural': 34630,
         'USB': 34032,
         'Soft': 33421,
         'Bag': 32285,
         'Water': 30940,
         'Baby': 30818,
         'Compatible': 30282,
         'Home': 29339,
         'Size': 28873,
         'iPhone': 28314,
         'Men': 28222,
         'Light': 28150,
         'Steel': 28098,
         'High': 27035,
         'Kit': 26673,
         'Food': 26205,
         'Cover': 26169,
         'Large': 25877,
         'Organic': 25412,
         'Cable': 25196,
         'Dry': 24759,
         'Premium': 24202,
         'Stainless': 23733,
         'Inch': 23707,
         'Sto

In [14]:
# find most common item sold
nouns_count_df = pd.DataFrame(list(nouns_count.items()), columns=['Word', 'Count'])
nouns_count_df = nouns_count_df.sort_values(by='Count', ascending=False)
nouns_count_df.to_csv('noun_counts.csv', index=False)
nouns_count_df.head(30)

,Word,Count
185,Pack,86249
65,Black,65305
106,Count,57772
557,Set,53579
155,Women,49854
184,Ounce,49240
808,Hair,42933
74,Gift,42417
5,Card,40799
487,Free,39532


In [16]:
# filter for most common item sold: USB and ensure its category is consistent
usb_df = df[df['Title'].str.contains("USB", case=False, na=False)]
category_counts = usb_df['Category'].value_counts().reset_index()
category_counts
usb_df = usb_df[usb_df['Category'].isin(["Electronics", "Computer & Accessories"])]
usb_df.to_csv('usb.csv', index=False)
usb_df


,Purchase Price Per Unit,Quantity,Title,ASIN/ISBN (Product Code),Category
5,10.99,1,Micro USB Cable Android Charger - Syncwire [2-...,B01GFB2E9M,Computer & Accessories
6,4.99,1,Amazon Basics USB 2.0 Charger Cable - A-Male t...,B00NH13S44,Computer & Accessories
17,26.77,1,Logitech M570 Wireless Trackball Mouse – Ergon...,B0043T7FXE,Computer & Accessories
26,139.99,1,"WD 8TB Elements Desktop External Hard Drive, U...",B07D5V2ZXD,Computer & Accessories
30,19.99,1,Anker Powerline II USB-C to USB-C 3.1 Gen 2 Ca...,B072JYDQ7N,Electronics
...,...,...,...,...,...
1391547,8.99,1,"JSAUX Micro USB Cable Android Charger, (2-Pack...",B07H91JTCD,Electronics
1391724,23.79,1,"Wireless Keyboard and Mouse, WisFox Full-Size ...",B088WV74FS,Computer & Accessories
1391918,21.99,1,"Anker Portable Charger, 313 Power Bank (PowerC...",B07QXV6N1B,Electronics
1391920,7.99,1,COSOOS 2 Short USB-C to iPhone Cables (10in/26...,B09CZ2KZ9P,Electronics


In [17]:
# prepare data for analysis of historic sales

usb_df["Demand Indicator"] = usb_df.groupby('ASIN/ISBN (Product Code)')['Quantity'].transform('sum')


## Model

In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# define features and targets
X = usb_df[['Quantity', 'Demand Indicator', "Category"]]  
y = usb_df['Purchase Price Per Unit']

# split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [20]:
# Define categorical and numerical features
categorical_features = X.select_dtypes(
   include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
   include=["float64", "int64"]
).columns.tolist()

In [21]:
preprocessor = ColumnTransformer(
   transformers=[
       ("cat", OneHotEncoder(), categorical_features),
       ("num", StandardScaler(), numerical_features),
   ]
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("regressor", GradientBoostingRegressor(random_state=42)),
    ]
)

In [22]:
# Perform 5-fold cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
print(f"Cross-Validation MAE: {-np.mean(cv_scores)}")

# Fit the model on the training data
pipeline.fit(X_train, y_train)

# Predict on the test set
y_pred = pipeline.predict(X_test)



Cross-Validation MAE: 21.58181645669986


In [23]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Test MAE: {mae}, Test RMSE: {rmse}")

Test MAE: 22.5596068955129, Test RMSE: 68.53113666889747


In [ ]:
results = pd.DataFrame({"Actual": y_test, "Predicted": y_pred})
results.to_csv("result/predictions.csv", index=False)